# 07: Interactive 3D domain/niche export

Read annotated three-dimensional predictions and export an interactive, confidence-aware volume view.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
SEED = 29
MODE = "path"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
ANNOTATED = ARTIFACTS / "annotated_volume.h5ad"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
MAX_POINTS = 100000
DISPLAY_GROUP_KEY = None
DISPLAY_GROUP_VALUE = None
USE_NUCLEUS_PATH = False
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )
NICHE_COLORS = {
    "Tumor nest": "#E31A1C",
    "Invasive Tumor nest": "#FF7F00",
    "Invasive Tumor region": "#7F0000",
    "Gastric-type Tumor": "#FB9A99",
    "Epithelium": "#F781BF",
    "Chief cell enriched": "#FCCDE5",
    "Neck cell enriched": "#DD1C77",
    "Inflamed gastric mucosal": "#980043",
    "Homeostatic stroma": "#66C2A4",
    "Gland stroma": "#1FBFCF",
    "Smooth muscle": "#1D91C0",
    "Fibrotic stroma": "#225EA8",
    "CAF enriched": "#00A087",
    "Plasma enriched stroma": "#7FCDBB",
    "Inflammation-infiltrated stroma": "#5DF3AD",
    "Normal vasculature": "#C51B7D",
    "Tumor vasculature": "#7A0177",
    "TLS-like structure": "#B8E186",
    "TAM enriched": "#4DAF4A",
    "TAN & Mono enriched": "#006D2C",
    "Low quality": "#BDBDBD",
    "Unknown": "#D9D9D9",
}
DOMAIN_COLORS = {
    "Tumor Parenchyma": "#E41A1C",
    "Tumor-stromal interface": "#984EA3",
    "Stroma": "#377EB8",
    "Adjacent Normal area": "#4DAF4A",
    "Low-quality": "#999999",
    "Unknown": "#D9D9D9",
}
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.visualization import render_volume

Select the annotated real volume produced by the preceding step.

In [ ]:
input_path = ANNOTATED
if not input_path.is_file():
    raise FileNotFoundError(f"Annotate the reconstruction first: {input_path}")

## 2. Render and save the interactive volume

Read labels and confidence values, sample display points by depth and annotation, and save a standalone Plotly HTML file. Display sampling never changes the underlying predictions.

Read the annotated volume, select a depth/label-stratified display sample, and export a self-contained interactive HTML view.

In [ ]:
html_path = require_new(OUT / "niche_domain_3d.html")
render_volume(
    input_path=input_path,
    output_path=html_path,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
    group_key=DISPLAY_GROUP_KEY,
    group_value=DISPLAY_GROUP_VALUE,
    max_points=MAX_POINTS,
    seed=SEED,
    points_per_stratum=60,
    domain_colors=DOMAIN_COLORS,
    niche_colors=NICHE_COLORS,
)
assert html_path.stat().st_size > 1000
print(html_path)